# Fabric Item Downloader v1

_Thin orchestrator over `fabric-downloader==0.1.1`._

Five cells:
1. **Install** the downloader wheel.
2. **Configure** the run (the only cell most users touch).
3. **Probe** the resolved write target + (optionally) the API endpoints.
4. **Run** the download — produces files under `Files/<output_root>/<run_label>/`
   and appends one row per item to the manifest Delta table.
5. **Explore** the manifest — display the five rollups.

All downloader logic lives in the installable wheel; edits to fetch
behavior, schema, or rollups happen there, not in this notebook.

In [ ]:
# Install the downloader wheel (re-run when you bump the version).
%pip install -q fabric-downloader==0.1.1

## 1. Configuration

`DownloaderConfig` is a frozen dataclass — every knob has a
sensible default. The most common edits are commented inline.

**Multi-type runs**: list every Fabric item type you want backed
up in `item_types`. Default `format_by_type` maps `Notebook` ->
`ipynb` (one self-contained file per notebook); every other type
falls back to parts mode (the API returns every definition part
as a separate file — `.platform`, `pipeline-content.json`,
`mashup.pq`, etc.). Override per-type with `format_by_type` if
you want parts mode for notebooks too.

In [ ]:
from fabric_downloader import DownloaderConfig

cfg = DownloaderConfig(
    # --- What to download ---
    item_types     = ("Notebook", "DataPipeline", "Dataflow"),
    # format_by_type = {"Notebook": "ipynb"},  # uncomment to override

    # --- Enumeration scope ---
    admin_mode         = True,   # tenant admin APIs when available
    read_workspace_ids = (),     # () = all visible workspaces
    max_items          = 0,      # 0 = no cap (set to 5 for dry-run)

    # --- Output ---
    output_root    = "fabric_item_backups",
    run_label      = "",        # "" -> auto yyyy-mm-dd_HH-MM-SS UTC
    manifest_table = "fabric_download_manifest",
    skip_existing  = True,       # restart-safe re-runs
    group_by_type  = True,       # subfolder per item type
    include_raw_definition = False,  # also save the .item.json envelope

    # --- Write target ---
    write_to_default_lakehouse = True,
    # write_workspace_id = "<ws-guid>",
    # write_lakehouse_id = "<lh-guid>",

    # --- Spark distribution ---
    num_partitions       = 0,    # 0 -> sc.defaultParallelism
    executor_concurrency = 30,
    max_retries          = 4,
)
cfg

## 2. Diagnostics — probe the resolved target

Prints the resolved write paths + the planned export mode per
item type. Pass a token to also probe Fabric REST endpoints.

In [ ]:
from fabric_downloader import resolve_paths, probe

resolved = resolve_paths(cfg)
probe(cfg, resolved)   # add token=<bearer> to probe APIs

## 3. Run the download

Enumerates items, distributes them across the Spark cluster, and
downloads them in parallel. Files land under
`Files/<output_root>/<run_label>/<wsName>__<wsId>/<Type>/`, and
the manifest is **appended** to the Delta table so you can
accumulate multiple runs over time (filter by `run_label`).

In [ ]:
from fabric_downloader.spark import run

result = run(cfg, spark)

print(f"Items attempted   : {result.item_count:,}")
print(f"Manifest table    : {result.manifest_table}")
print()
print("By status:")
for k, n in sorted(result.by_status.items()):
    print(f"  {k:18s}  {n:>6,}")
print()
print("By item type:")
for k, n in sorted(result.by_type.items()):
    print(f"  {k:18s}  {n:>6,}")

## 4. Explore the manifest

Each rollup is a regular Spark DataFrame — display, filter, or
join as you like. See `fabric_downloader.persist.SummaryReport`
for the full list.

In [ ]:
display(result.summary.by_status)

In [ ]:
display(result.summary.by_type)

In [ ]:
display(result.summary.by_workspace)

In [ ]:
display(result.summary.errors)

In [ ]:
display(result.summary.balance_check)

In [ ]:
display(result.summary.run_summary)